<a href="https://colab.research.google.com/github/AnjanVankayala/News-Headlines-Aggregator/blob/main/News_Headlines_Aggregator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4

In [25]:
import json

# --- Configuration Data ---

config_data = {
    "bbc": {
        "url": "https://www.bbc.com/news",
        "base_url": "https://www.bbc.com",
        "selector": "h2"
    },
    "reuters_world": {
        "url": "https://www.reuters.com/world/",
        "base_url": "https://www.reuters.com",
        "selector": "a[data-testid='Link']"
    },
    "times_of_india": {
        "url": "https://timesofindia.indiatimes.com/news",
        "base_url": "https://timesofindia.indiatimes.com",
         "selector": "p.CRKrj"
     },
    # Add more sources here following the same structure
    "ndtv": {
        "url": "https://www.ndtv.com/latest",
        "base_url": "https://www.ndtv.com",
        "selector": "a.NwsLstPg_ttl-lnk"
    },
    "the_hindu": {
        "url": "https://www.thehindu.com/news/",
        "base_url": "https://www.thehindu.com",
        "selector": "a"
    },
    "cnn_world": {
        "url": "https://edition.cnn.com/world",
        "base_url": "https://edition.cnn.com",
        "selector": "span"
    },
    "the_guardian_world": {
        "url": "https://www.theguardian.com/world",
        "base_url": "https://www.theguardian.com",
        "selector": "a"
    }
}

config_file_path = 'config.json'
try:
    with open(config_file_path, 'w') as f:
        json.dump(config_data, f, indent=4)
    print(f"Configuration file '{config_file_path}' created successfully with updated selectors.")
except Exception as e:
    print(f"Error writing configuration file: {e}")

Configuration file 'config.json' created successfully with updated selectors.


In [17]:
import requests
from bs4 import BeautifulSoup
import json
import argparse
from urllib.parse import urljoin # To handle relative URLs

# --- Constants ---
CONFIG_FILE = 'config.json'
# Add a User-Agent header to mimic a browser request
REQUEST_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9'
}


def load_config(filepath=CONFIG_FILE):
    """Loads the configuration data from the JSON file."""
    try:
        with open(filepath, 'r') as f:
            config = json.load(f)
        print("Configuration loaded successfully.")
        return config
    except FileNotFoundError:
        print(f"Error: Configuration file '{filepath}' not found.")
        return None
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{filepath}'. Check its format.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while loading config: {e}")
        return None

def scrape_source(source_id, source_config):
    """Scrapes headlines and links from a single news source."""
    url = source_config.get('url')
    base_url = source_config.get('base_url')
    selector = source_config.get('selector')

    if not all([url, base_url, selector]):
        print(f"Error: Incomplete configuration for source '{source_id}'. Skipping.")
        return []

    print(f"\nFetching headlines from {source_id.upper()} ({url})...")

    headlines = []
    try:
        response = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        headline_elements = soup.select(selector)

        if not headline_elements:
            print(f"Warning: No elements found for '{source_id}' using selector '{selector}'. Website structure might have changed or selector needs refinement.")
            return []

        print(f"Found {len(headline_elements)} potential elements using selector for {source_id.upper()}. Processing...")

        processed_links = set()

        for element in headline_elements:
            title = element.get_text(strip=True)
            link = None
            absolute_link = None

            # Find the associated link (href)
            link_element = None
            if element.name == 'a':
                link_element = element
            else:
                link_element = element.find_parent('a')

            if link_element:
                link = link_element.get('href')

            # Ensure we have a title and a link, and construct absolute URL
            if title and link:
                absolute_link = urljoin(base_url, link)

                if absolute_link not in processed_links:
                    headlines.append({'title': title, 'link': absolute_link})
                    processed_links.add(absolute_link)

        print(f"Successfully processed {len(headlines)} headlines from {source_id.upper()}.")

    except requests.exceptions.RequestException as e:
        # Specific check for 401/403 errors
        if e.response is not None and e.response.status_code in [401, 403]:
             print(f"Error fetching data from {source_id.upper()}: Status Code {e.response.status_code} (Forbidden/Unauthorized). Access may be blocked.")
        else:
             print(f"Error fetching data from {source_id.upper()}: {e}")
    except Exception as e:
        print(f"An error occurred during scraping {source_id.upper()}: {e}")

    return headlines

def display_headlines(headlines_by_source, keyword_filter=None):
    """Formats and displays the aggregated headlines."""
    print("\n--- Aggregated News Headlines ---")
    overall_count = 0
    filtered_count = 0

    if not headlines_by_source:
        print("No headlines were successfully scraped.")
        return

    for source_id, headlines in headlines_by_source.items():
        if not headlines:
            continue

        print(f"\n--- {source_id.upper()} ---")
        source_headline_count = 0
        for i, headline in enumerate(headlines):
            title = headline.get('title', 'N/A')
            link = headline.get('link', '#')

            if keyword_filter and keyword_filter.lower() not in title.lower():
                continue

            overall_count += 1
            source_headline_count += 1
            print(f"{overall_count}. {title}")
            print(f"   Link: {link}")

        if source_headline_count == 0 and keyword_filter:
            print(f"(No headlines matching '{keyword_filter}' found for this source)")
        elif source_headline_count == 0:
             print("(No headlines retrieved for this source)")


        filtered_count += source_headline_count

    print("\n--- End of Headlines ---")
    if keyword_filter:
        print(f"Displayed {filtered_count} headlines matching '{keyword_filter}'.")
    else:
         print(f"Displayed a total of {overall_count} headlines.")



In [18]:
import json

def main(sources_str=None, filter_keyword=None):
    """
    Main function to run the news aggregator.

    Args:
        sources_str (str, optional): Comma-separated string of source IDs
                                     (e.g., 'bbc,reuters_world').
                                     Defaults to None (uses all sources in config).
        filter_keyword (str, optional): Keyword to filter headlines by (case-insensitive).
                                        Defaults to None (no filtering).
    """
    print("\n--- Starting News Aggregator ---")
    print(f"Parameters: sources='{sources_str}', filter='{filter_keyword}'")

    # Load configuration
    config = load_config()
    if not config:
        print("Exiting due to configuration error.")
        return

    # Determine which sources to scrape
    source_ids_to_scrape = []
    if sources_str:
        provided_source_ids = [s.strip().lower() for s in sources_str.split(',')]
        # Validate source IDs against the loaded config
        for sid in provided_source_ids:
            if sid in config:
                source_ids_to_scrape.append(sid)
            else:
                print(f"Warning: Provided source ID '{sid}' not found in configuration. Skipping.")
    else:
        source_ids_to_scrape = list(config.keys())
        print("No specific sources provided, using all sources from config.")


    if not source_ids_to_scrape:
        print("No valid sources selected or configured. Exiting.")
        return

    print(f"Selected sources: {', '.join(source_ids_to_scrape)}")

    # Scrape headlines from selected sources
    all_headlines = {}
    for source_id in source_ids_to_scrape:
        source_config = config.get(source_id)
        if source_config:
             all_headlines[source_id] = scrape_source(source_id, source_config)

    # Display the results
    display_headlines(all_headlines, filter_keyword)
    print("\n--- Aggregator Finished ---")

# --- Example Usage ---
# To run with specific sources and a filter:
# main(sources_str='bbc,times_of_india', filter_keyword='technology')

# To run with only one source, no filter:
# main(sources_str='bbc')

# To run with all configured sources, but apply a filter:
# main(filter_keyword='world')

# To run with all configured sources and no filter:
# main()

In [27]:
main()


--- Starting News Aggregator ---
Parameters: sources='None', filter='None'
Configuration loaded successfully.
No specific sources provided, using all sources from config.
Selected sources: bbc, reuters_world, times_of_india, ndtv, the_hindu, cnn_world, the_guardian_world

Fetching headlines from BBC (https://www.bbc.com/news)...
Found 63 potential elements using selector for BBC. Processing...
Successfully processed 35 headlines from BBC.

Fetching headlines from REUTERS_WORLD (https://www.reuters.com/world/)...
Error fetching data from REUTERS_WORLD: Status Code 401 (Forbidden/Unauthorized). Access may be blocked.

Fetching headlines from TIMES_OF_INDIA (https://timesofindia.indiatimes.com/news)...
Found 59 potential elements using selector for TIMES_OF_INDIA. Processing...
Successfully processed 54 headlines from TIMES_OF_INDIA.

Fetching headlines from NDTV (https://www.ndtv.com/latest)...
Error fetching data from NDTV: Status Code 403 (Forbidden/Unauthorized). Access may be blocke